In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print(f"PyTorch : {torch.__version__}")
print(f"Device     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [2]:
!pip install -q huggingface_hub

import os
from google.colab import userdata
from huggingface_hub import login

# Lấy token từ Secrets của Colab
hf_token = userdata.get('tuankun08')

# Tiến hành đăng nhập
login(token=hf_token)
print("Đã kết nối thành công với Hugging Face!")

Đã kết nối thành công với Hugging Face!


In [3]:
from huggingface_hub import snapshot_download, login
from google.colab import userdata

repo_id="tuankun08/Dataset"

# Tải toàn bộ repository/folder về thư mục '/content/Dataset' trên Colab
snapshot_download(
    repo_id=repo_id,
    allow_patterns="Dataset/*",
    repo_type="dataset",
    local_dir="/content/Dataset"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

'/content/Dataset'

In [4]:
!git clone https://github.com/tuananhh08/Hallsensor_final.git

Cloning into 'Hallsensor_final'...
remote: Enumerating objects: 2363, done.
remote: Counting objects: 100% (205/205), done.
remote: Compressing objects: 100% (182/182), done.
remote: Total 2363 (delta 38), reused 172 (delta 20), pack-reused 2158 (from 1)
Receiving objects: 100% (2363/2363), 143.38 MiB | 20.96 MiB/s, done.
Resolving deltas: 100% (407/407), done.


In [5]:
!pip install optuna --quiet
!pip install -q scikit-learn pandas numpy

DATA_DIR  = "/content/Dataset/Dataset"
REPO      = "/content/Hallsensor_final/Benchmark/AmagPoseNet"
RUNS      = "/content/drive/MyDrive/AmagPoseNet_runs"
CACHE     = "/content/amagposenet_cache"

# Finetune data
RAW_VOLTAGE           = f"{DATA_DIR}/Grid_add_Random_data.csv"
CLEAN_VOLTAGE         = f"{DATA_DIR}/Grid_add_Random_data_computed.csv"

# Grid data
GRID_VOLTAGE          = f"{DATA_DIR}/Grid_data.csv"
GRID_CLEAN_VOLTAGE    = f"{DATA_DIR}/Grid_data_computed.csv"

# Helix data
HELIX_VOLTAGE         = f"{DATA_DIR}/Helix_data_2.csv"
HELIX_CLEAN_VOLTAGE   = f"{DATA_DIR}/Helix_data_2_computed.csv"

# Random data
RANDOM_VOLTAGE        = f"{DATA_DIR}/Random_data.csv"
RANDOM_CLEAN_VOLTAGE  = f"{DATA_DIR}/Random_data_computed.csv"

# Label
RAW_LABEL             = f"{DATA_DIR}/Grid_add_Random_points_coordinates.csv"
GRID_LABEL            = f"{DATA_DIR}/Grid_points_coordinates.csv"
HELIX_LABEL           = f"{DATA_DIR}/Helix_points_coordinates_2.csv"
RANDOM_LABEL          = f"{DATA_DIR}/Random_points_coordinates.csv"

# Synthetic data ~700k data
SYNTHETIC_VOLTAGE = f"{DATA_DIR}/synthetic_grid_data.csv"
SYNTHETIC_LABEL   = f"{DATA_DIR}/synthetic_grid_coordinates.csv"

# Calibration parameters path
CALIB_PHYSICAL_CSV = f"{DATA_DIR}/Calibration_Parameters_r_per_sensor.csv"
CALIB_ALPHA_CSV    = f"{DATA_DIR}/Calibration_Alpha_r_per_sensor.csv"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 27.5 MB/s eta 0:00:00


In [6]:
# PRE-TRAIN CALINET
!python $REPO/run.py \
  --stage calinet \
  --raw-voltage "$RAW_VOLTAGE" \
  --clean-voltage "$CLEAN_VOLTAGE" \
  --raw-label "$RAW_LABEL" \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label "$SYNTHETIC_LABEL" \
  --ckpt-dir "$RUNS/calinet" \
  --cache-dir "$CACHE" \
  --batch-size 64 \
  --num-epochs 260 \
  --lr 0.0007975614143764486 \
  --weight-decay 3.281437096954538e-05 \
  --warmup-epochs 5 \
  --eta-min 1e-6 \
  --patience 45 \
  --save-every 10 \
  --grad-clip-norm 1.0 \
  --calib-reduction mean \
  --no-compile

[cache] Loading CSVs ...
[cache] Fitting scalers on training splits ...
[cache] Writing NPY arrays ...
[cache] Done. raw=3,222  synth=691,200  cache=/content/amagposenet_cache
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

  Stage 1 – CaliNet Training
  Train : 2,578  Val: 644
  Device: cuda  LR=0.0007975614143764486  Epochs=260

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what thi

In [ ]:
#PRE-TRAIN POSENET
!python $REPO/run.py \
  --stage posenet \
  --raw-voltage "$RAW_VOLTAGE" \
  --clean-voltage "$CLEAN_VOLTAGE" \
  --raw-label "$RAW_LABEL" \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label "$SYNTHETIC_LABEL" \
  --ckpt-dir "$RUNS/posenet" \
  --cache-dir "$CACHE" \
  --batch-size 512 \
  --num-epochs 400 \
  --lr 0.00048438278642785697 \
  --weight-decay 6.235377135673162e-06 \
  --warmup-epochs 5 \
  --eta-min 1e-6 \
  --beta 0.2234016585319006 \
  --lambda-pos 0.015299279685163814 \
  --patience 75 \
  --save-every 10 \
  --grad-clip-norm 1.0 \
  --no-compile

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()

  Stage 2 – PoseNet Pre-training (synthetic data)
  Train : 552,960  Val: 138,240
  Device: cuda  LR=0.00048438278642785697  beta=2234016585319006.0  Epochs=400

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even f

In [ ]:
#FINE-TUNE END-TO-END
!python $REPO/run.py \
  --stage finetune \
  --raw-voltage "$RAW_VOLTAGE" \
  --clean-voltage "$CLEAN_VOLTAGE" \
  --raw-label "$RAW_LABEL" \
  --synthetic-voltage "$SYNTHETIC_VOLTAGE" \
  --synthetic-label "$SYNTHETIC_LABEL" \
  --calinet-checkpoint "$RUNS/calinet/calinet_pretrained.pt" \
  --posenet-checkpoint "$RUNS/posenet/posenet_pretrained.pt" \
  --ckpt-dir "$RUNS/finetune" \
  --cache-dir "$CACHE" \
  --batch-size 64 \
  --num-epochs 260 \
  --lr-calinet 9.024521230854463e-05 \
  --lr-posenet 0.002009188725553257 \
  --weight-decay 1.3005157012713626e-06 \
  --warmup-epochs 5 \
  --eta-min 1e-6 \
  --beta 0.012111972525642022 \
  --lambda-pos 0.016592359959339182 \
  --lambda-calib 0.023807493889028322 \
  --calib-reduction mean \
  --patience 50 \
  --save-every 5 \
  --grad-clip-norm 1.0 \
  --no-compile

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[Stage3] Loaded CaliNet  from: /content/drive/MyDrive/AmagPoseNet_runs/calinet/calinet_pretrained.pt
[Stage3] Loaded PoseNet  from: /content/drive/MyDrive/AmagPoseNet_runs/posenet/posenet_pretrained.pt

  Stage 3 – End-to-end Fine-tuning
  Train : 2,578  Val: 644
  Device: cuda  LR_cal=9.024521230854463e-05  LR_pose=0.002009188725553257
  beta=0.012111972525642022  lambda_calib=0.023807493889028322  Epochs=260

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader w

In [10]:
# Test Inference
import os
os.makedirs(f"{RUNS}/evaluation", exist_ok=True)

!python $REPO/evaluation/evaluate.py \
  --test_voltage "$HELIX_CLEAN_VOLTAGE" \
  --test_label "$HELIX_LABEL" \
  --ckpt_dir "$RUNS/finetune" \
  --checkpoint "full_model_best.pt" \
  --scalers "$RUNS/finetune/scalers.pkl" \
  --batch-size 64 \
  --device cuda \
  --no-compile \
  --out "$RUNS/evaluation/amagposenet_trajectory.png" \
  --out-csv "$RUNS/evaluation/amagposenet_predictions.csv"

  AMagPoseNet Evaluation
  Voltage : /content/Dataset/Dataset/Helix_data_2_computed.csv
  Label   : /content/Dataset/Dataset/Helix_points_coordinates_2.csv
  Ckpt    : /content/drive/MyDrive/AmagPoseNet_runs/finetune

  Test samples: 150
  Loading scalers from /content/drive/MyDrive/AmagPoseNet_runs/finetune/scalers.pkl ...
  Device  : cuda
  Loading checkpoint: /content/drive/MyDrive/AmagPoseNet_runs/finetune/full_model_best.pt ...
  Checkpoint epoch : 243
  Best val loss    : 0.002254

Running inference ...

  Pt          PX       PY       PZ       GX       GY       GZ  PosErr(mm)  OriErr(°)
  -----------------------------------------------------------------------------------------------
  0       0.0953   0.7059  -0.0602   0.0950   0.7050  -0.0635        3.47      8.245
  1       0.0953   0.7191  -0.0620   0.0932   0.7196  -0.0633        2.53     11.500
  2       0.0949   0.7322  -0.0635   0.0884   0.7335  -0.0631        6.60      9.157
  3       0.0924   0.7470  -0.0638   0.0809   